# OneFormer windowpane 감지 테스트 (Colab)

`shi-labs/oneformer_ade20k_swin_large`가 실내 사진에서 ADE20K 클래스 **windowpane**을 얼마나 잘 잡는지 확인합니다.
GT 마스크가 없으므로 시각적 오버레이와 정량 지표(면적 비율, 평균 확률, 인스턴스 수·점수, 추론 시간)로 평가합니다.

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 위에서부터 순서대로 실행하세요.

In [ ]:
# 1. 패키지 설치 (transformers는 최신으로, AVIF 읽기용 플러그인 포함)
!pip -q install -U transformers ftfy pillow-avif-plugin

In [ ]:
# 2. 임포트와 설정
import os, glob, time, shutil
import numpy as np
import torch
import torch.nn.functional as F
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
try:
    import pillow_avif  # noqa: F401  (Pillow가 AVIF를 기본 지원하지 않는 경우 대비)
except ImportError:
    pass
from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation

MODEL_ID = "shi-labs/oneformer_ade20k_swin_large"
IMG_DIR = "window_test_pics"
OUT_DIR = "window_test_results"
MAX_SIDE = 1024          # 긴 변이 이보다 크면 축소
INST_THRESHOLD = 0.5     # instance 세그먼트 점수 임계값
CC_MIN_AREA = 0.001      # semantic 마스크 연결요소 최소 면적(이미지 대비 비율)
EXTS = (".jpg", ".jpeg", ".png", ".avif", ".webp")
CONFUSABLE = ["door", "curtain", "mirror", "screen door", "blind", "painting"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 3. 테스트 이미지 업로드
#    window_test_pics 폴더가 비어 있으면 업로드 창이 뜹니다.
#    (Google Drive에 있다면 아래 주석을 풀고 IMG_DIR을 Drive 경로로 바꿔도 됩니다)
# from google.colab import drive; drive.mount('/content/drive'); IMG_DIR = '/content/drive/MyDrive/window_test_pics'
from google.colab import files

if not [f for f in os.listdir(IMG_DIR) if f.lower().endswith(EXTS)]:
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(IMG_DIR, name), "wb") as f:
            f.write(data)
        if os.path.exists(name):
            os.remove(name)

img_paths = sorted(p for p in glob.glob(os.path.join(IMG_DIR, "*")) if p.lower().endswith(EXTS))
print(f"{len(img_paths)}장:", [os.path.basename(p) for p in img_paths])

In [ ]:
# 4. 모델 로드 (약 0.9GB 다운로드)
processor = OneFormerProcessor.from_pretrained(MODEL_ID)
model = OneFormerForUniversalSegmentation.from_pretrained(MODEL_ID).to(device).eval()

id2label = {int(k): v.strip() for k, v in model.config.id2label.items()}

def find_label(name):
    for i, l in id2label.items():          # 정확히 일치
        if l == name:
            return i
    for i, l in id2label.items():          # "windowpane, window" 같은 형태 대비
        if l.startswith(name):
            return i
    return None

WIN_ID = find_label("windowpane")
if WIN_ID is None:
    WIN_ID = find_label("window")
assert WIN_ID is not None, "windowpane 클래스를 찾지 못함"
CONF_IDS = {n: find_label(n) for n in CONFUSABLE if find_label(n) is not None}
print(f"windowpane id = {WIN_ID} ({id2label[WIN_ID]!r})")
print("혼동 후보:", {n: i for n, i in CONF_IDS.items()})

In [ ]:
# 5. 함수 정의
def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    s = MAX_SIDE / max(w, h)
    if s < 1:
        img = img.resize((round(w * s), round(h * s)), Image.LANCZOS)
    return img

@torch.inference_mode()
def run_task(img, task):
    inputs = processor(images=img, task_inputs=[task], return_tensors="pt").to(device)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model(**inputs)
    if device == "cuda":
        torch.cuda.synchronize()
    return out, time.perf_counter() - t0

@torch.inference_mode()
def semantic_maps(out, size_hw):
    """post_process_semantic_segmentation과 같은 방식으로 클래스별 점수맵을 만든 뒤
    argmax 맵과 windowpane 확률맵(클래스 합으로 정규화)을 반환"""
    cls = out.class_queries_logits.softmax(-1)[..., :-1]      # (1, Q, C)  마지막 = no-object
    msk = out.masks_queries_logits.sigmoid()                  # (1, Q, h, w)
    seg = torch.einsum("bqc,bqhw->bchw", cls, msk)
    seg = F.interpolate(seg, size=size_hw, mode="bilinear", align_corners=False)[0]  # (C, H, W)
    argmax = seg.argmax(0)
    win_prob = seg[WIN_ID] / seg.sum(0).clamp_min(1e-8)
    return argmax.cpu().numpy(), win_prob.cpu().numpy()

def instance_windows(out, size_hw):
    res = processor.post_process_instance_segmentation(
        out, threshold=INST_THRESHOLD, target_sizes=[size_hw])[0]
    seg = res.get("segmentation")
    if seg is None:
        return []
    seg = seg.cpu().numpy() if torch.is_tensor(seg) else np.asarray(seg)
    wins = []
    for info in res["segments_info"]:
        if info["label_id"] != WIN_ID:
            continue
        m = seg == info["id"]
        if not m.any():
            continue
        ys, xs = np.where(m)
        wins.append({"score": float(info["score"]),
                     "box": (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1),
                     "area": int(m.sum())})
    return wins

def cc_boxes(mask):
    """semantic windowpane 마스크를 연결요소로 쪼개 박스로 만듦 (붙은 창문은 하나로 나올 수 있음)"""
    H, W = mask.shape
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    return [(x, y, x + w, y + h) for x, y, w, h, a in stats[1:] if a >= CC_MIN_AREA * H * W]

def draw_boxes(ax, boxes, color, labels=None):
    for i, (x0, y0, x1, y1) in enumerate(boxes):
        ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=color, lw=2))
        if labels:
            ax.text(x0, y0 - 4, labels[i], color="black", fontsize=8,
                    bbox=dict(fc=color, ec="none", pad=1))

def visualize(name, img, win_mask, win_prob, cc, wins, path):
    arr = np.asarray(img)
    overlay = arr.copy()
    overlay[win_mask] = (0.45 * overlay[win_mask] + 0.55 * np.array([255, 0, 0])).astype(np.uint8)

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    axes[0].imshow(arr); axes[0].set_title("Original")
    axes[1].imshow(overlay); draw_boxes(axes[1], cc, "yellow")
    axes[1].set_title(f"Semantic windowpane (+CC boxes: {len(cc)})")
    im = axes[2].imshow(win_prob, cmap="inferno", vmin=0, vmax=1)
    axes[2].set_title("windowpane probability"); fig.colorbar(im, ax=axes[2], fraction=0.035)
    axes[3].imshow(arr); draw_boxes(axes[3], [w["box"] for w in wins], "lime",
                                    [f'{w["score"]:.2f}' for w in wins])
    axes[3].set_title(f"Instance windows (thr={INST_THRESHOLD}): {len(wins)}")
    for a in axes:
        a.axis("off")
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.show()
    plt.close(fig)

In [ ]:
# 6. 실행
# 첫 추론은 CUDA 초기화로 느리므로, 시간 측정 전에 한 번 워밍업
if img_paths:
    _ = run_task(load_image(img_paths[0]), "semantic")

rows = []
for p in img_paths:
    name = os.path.splitext(os.path.basename(p))[0]
    try:
        img = load_image(p)
    except Exception as e:
        print(f"[skip] {p}: {e}")
        continue
    size_hw = (img.height, img.width)

    sem_out, t_sem = run_task(img, "semantic")
    argmax, win_prob = semantic_maps(sem_out, size_hw)
    win_mask = argmax == WIN_ID
    cc = cc_boxes(win_mask)

    inst_out, t_inst = run_task(img, "instance")
    wins = instance_windows(inst_out, size_hw)

    row = {
        "image": os.path.basename(p),
        "size(WxH)": f"{img.width}x{img.height}",
        "window_%": round(100 * win_mask.mean(), 2),
        "window_mean_prob": round(float(win_prob[win_mask].mean()), 3) if win_mask.any() else np.nan,
        "cc_boxes": len(cc),
        "inst_count": len(wins),
        "inst_scores": ", ".join(f'{w["score"]:.2f}' for w in sorted(wins, key=lambda w: -w["score"])),
        "t_semantic_s": round(t_sem, 2),
        "t_instance_s": round(t_inst, 2),
    }
    for n, cid in CONF_IDS.items():
        row[f"{n}_%"] = round(100 * (argmax == cid).mean(), 2)
    rows.append(row)

    visualize(name, img, win_mask, win_prob, cc, wins, os.path.join(OUT_DIR, f"{name}_vis.png"))

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False, encoding="utf-8-sig")
df

In [ ]:
# 7. 결과 다운로드 (시각화 PNG + summary.csv)
shutil.make_archive("window_test_results", "zip", OUT_DIR)
files.download("window_test_results.zip")